# Observatorio de Educacion - Carga y Analisis de Datos

**Objetivo:** Cargar, explorar y analizar la calidad de los datos de cada fuente del Observatorio de Educacion de Cali.

**Fuentes de datos:**
1. Reporte de Matricula 2026
2. Indicadores de Eficiencia y Cobertura 2026
3. Estudiantes por Docente y Equipo de Computo 2026
4. Informacion Geografica de Sedes
5. Reporte de Inventario de Equipos de Computo
6. Intervenciones de Infraestructura (22 sedes, 49 sedes emprestito, historico obras)

**Metodologia:** Para cada fuente se analiza:
- Dimensiones (filas x columnas)
- Tipos de datos
- Valores nulos
- Valores vacios
- Registros duplicados
- Valores unicos por columna clave
- Estadisticas descriptivas

## Celda 0 - Configuracion del entorno

In [ ]:
import os
import sys

# Detectar si estamos en Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Clonar repo si no existe
    REPO_URL = 'https://github.com/j0rg3c45/Observatorio_de_Educacion.git'
    REPO_DIR = '/content/Observatorio_de_Educacion'
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)
    DATA_PATH = 'data/Fuentes de datos'
else:
    # Ejecucion local
    DATA_PATH = '../data/Fuentes de datos'

print(f'Entorno: {"Colab" if IN_COLAB else "Local"}')
print(f'Ruta de datos: {DATA_PATH}')
print(f'Directorio de trabajo: {os.getcwd()}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuracion de graficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Librerias cargadas correctamente (pandas, numpy, matplotlib, seaborn)')

## Funcion auxiliar de analisis de calidad

In [ ]:
def analisis_calidad(df, nombre_fuente):
    """Analiza la calidad de un DataFrame y muestra resumen completo."""
    print('=' * 70)
    print(f'ANALISIS DE CALIDAD: {nombre_fuente}')
    print('=' * 70)
    
    # Dimensiones
    print(f'\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
    
    # Tipos de datos
    print(f'\nTipos de datos:')
    print(df.dtypes.value_counts().to_string())
    
    # Valores nulos
    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df) * 100).round(2)
    nulos_df = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': nulos_pct})
    nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Nulos', ascending=False)
    
    if len(nulos_df) > 0:
        print(f'\nColumnas con valores nulos ({len(nulos_df)} de {df.shape[1]}):')
        print(nulos_df.to_string())
    else:
        print(f'\nValores nulos: NINGUNO')
    
    # Valores vacios (strings vacios)
    cols_obj = df.select_dtypes(include='object').columns
    if len(cols_obj) > 0:
        vacios = (df[cols_obj] == '').sum()
        vacios = vacios[vacios > 0]
        if len(vacios) > 0:
            print(f'\nColumnas con strings vacios:')
            print(vacios.to_string())
        else:
            print(f'\nStrings vacios: NINGUNO')
    
    # Duplicados
    duplicados = df.duplicated().sum()
    print(f'\nRegistros duplicados: {duplicados} ({duplicados/len(df)*100:.2f}%)')
    
    # Valores unicos por columna
    print(f'\nValores unicos por columna:')
    print(df.nunique().to_string())
    
    # Estadisticas descriptivas numericas
    cols_num = df.select_dtypes(include=[np.number]).columns
    if len(cols_num) > 0:
        print(f'\nEstadisticas descriptivas (columnas numericas):')
        print(df[cols_num].describe().to_string())
    
    print('\n')
    return nulos_df

## Funcion auxiliar de visualizacion

In [ ]:
def grafico_nulos(df, nombre_fuente, top_n=15):
    """Grafico de barras horizontales con las columnas con mas nulos."""
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0].sort_values(ascending=True).tail(top_n)
    
    if len(nulos) == 0:
        print(f'{nombre_fuente}: Sin valores nulos - no se genera grafico.')
        return
    
    fig, ax = plt.subplots(figsize=(9, max(3, len(nulos) * 0.4)))
    nulos.plot(kind='barh', ax=ax, color=sns.color_palette('Set2')[1])
    ax.set_xlabel('Cantidad de nulos')
    ax.set_title(f'Columnas con valores nulos - {nombre_fuente}')
    for i, v in enumerate(nulos.values):
        ax.text(v + 0.5, i, f'{v} ({v/len(df)*100:.1f}%)', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()


def grafico_tipos_datos(df, nombre_fuente):
    """Grafico de torta con la distribucion de tipos de datos."""
    tipos = df.dtypes.value_counts()
    fig, ax = plt.subplots(figsize=(6, 4))
    tipos.plot(kind='pie', ax=ax, autopct='%1.0f%%', startangle=90)
    ax.set_ylabel('')
    ax.set_title(f'Tipos de datos - {nombre_fuente}')
    plt.tight_layout()
    plt.show()


def grafico_distribucion_numericas(df, nombre_fuente, max_cols=6):
    """Histogramas de las primeras columnas numericas."""
    cols_num = df.select_dtypes(include=[np.number]).columns[:max_cols]
    if len(cols_num) == 0:
        return
    
    n_cols = min(3, len(cols_num))
    n_rows = (len(cols_num) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
    if n_rows * n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(cols_num):
        axes[i].hist(df[col].dropna(), bins=20, color=sns.color_palette('Set2')[0], edgecolor='white')
        axes[i].set_title(col, fontsize=10)
        axes[i].set_xlabel('')
    
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    
    fig.suptitle(f'Distribucion de variables numericas - {nombre_fuente}', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


print('Funciones de visualizacion cargadas')

---
## 1. Reporte de Matricula 2026

In [ ]:
# Cargar matricula
ruta_matricula = os.path.join(DATA_PATH, 'Reporte de matr\u00edcula', '01_Matricula_2026.xlsx')

# Fallback sin tilde
if not os.path.exists(ruta_matricula):
    ruta_matricula = os.path.join(DATA_PATH, 'Reporte de matricula', '01_Matricula_2026.xlsx')

print(f'Cargando: {ruta_matricula}')
df_matricula = pd.read_excel(ruta_matricula)
print(f'Carga exitosa.')
df_matricula.head()

In [ ]:
# Analisis de calidad - Matricula
nulos_matricula = analisis_calidad(df_matricula, 'REPORTE DE MATRICULA 2026')

In [ ]:
# Columnas del dataset
print('Columnas disponibles:')
for i, col in enumerate(df_matricula.columns, 1):
    print(f'  {i:2d}. {col} ({df_matricula[col].dtype})')

In [ ]:
# Graficos - Matricula 2026
grafico_nulos(df_matricula, 'Matricula 2026')
grafico_tipos_datos(df_matricula, 'Matricula 2026')
grafico_distribucion_numericas(df_matricula, 'Matricula 2026')

---
## 2. Indicadores de Eficiencia y Cobertura 2026

In [ ]:
# Cargar indicadores
ruta_indicadores = os.path.join(DATA_PATH, 'Indicadores de eficiencia y de cobertura', '02_Indicadores_2026.xlsx')
print(f'Cargando: {ruta_indicadores}')

df_indicadores = pd.read_excel(ruta_indicadores)
print(f'Carga exitosa.')
df_indicadores.head()

In [ ]:
# Analisis de calidad - Indicadores
nulos_indicadores = analisis_calidad(df_indicadores, 'INDICADORES DE EFICIENCIA Y COBERTURA 2026')

In [ ]:
# Columnas del dataset
print('Columnas disponibles:')
for i, col in enumerate(df_indicadores.columns, 1):
    print(f'  {i:2d}. {col} ({df_indicadores[col].dtype})')

In [ ]:
# Graficos - Indicadores 2026
grafico_nulos(df_indicadores, 'Indicadores 2026')
grafico_tipos_datos(df_indicadores, 'Indicadores 2026')
grafico_distribucion_numericas(df_indicadores, 'Indicadores 2026')

---
## 3. Estudiantes por Docente y Equipo de Computo 2026

In [ ]:
# Cargar docentes y equipos
ruta_docentes = os.path.join(DATA_PATH, 'Indicadores de docentes y equipos de computo', '03_Estudiantes_por_docente_y_equipo_2026.xlsx')
print(f'Cargando: {ruta_docentes}')

df_docentes = pd.read_excel(ruta_docentes)
print(f'Carga exitosa.')
df_docentes.head()

In [ ]:
# Analisis de calidad - Docentes y equipos
nulos_docentes = analisis_calidad(df_docentes, 'ESTUDIANTES POR DOCENTE Y EQUIPO 2026')

In [ ]:
# Columnas del dataset
print('Columnas disponibles:')
for i, col in enumerate(df_docentes.columns, 1):
    print(f'  {i:2d}. {col} ({df_docentes[col].dtype})')

In [ ]:
# Graficos - Docentes y Equipos 2026
grafico_nulos(df_docentes, 'Docentes y Equipos 2026')
grafico_tipos_datos(df_docentes, 'Docentes y Equipos 2026')
grafico_distribucion_numericas(df_docentes, 'Docentes y Equipos 2026')

---
## 4. Informacion Geografica de Sedes

In [ ]:
# Cargar info geografica
ruta_geo = os.path.join(DATA_PATH, 'Informaci\u00f3n geogr\u00e1fica sedes.xlsx')

# Fallback sin tildes
if not os.path.exists(ruta_geo):
    ruta_geo = os.path.join(DATA_PATH, 'Informacion geografica sedes.xlsx')

print(f'Cargando: {ruta_geo}')
df_geo = pd.read_excel(ruta_geo)
print(f'Carga exitosa.')
df_geo.head()

In [ ]:
# Analisis de calidad - Info geografica
nulos_geo = analisis_calidad(df_geo, 'INFORMACION GEOGRAFICA DE SEDES')

In [ ]:
# Columnas del dataset
print('Columnas disponibles:')
for i, col in enumerate(df_geo.columns, 1):
    print(f'  {i:2d}. {col} ({df_geo[col].dtype})')

In [ ]:
# Graficos - Info Geografica
grafico_nulos(df_geo, 'Info Geografica Sedes')
grafico_tipos_datos(df_geo, 'Info Geografica Sedes')

---
## 5. Reporte de Inventario de Equipos de Computo

In [ ]:
# Cargar inventario equipos
ruta_equipos = os.path.join(DATA_PATH, 'REPORTE DE INVENTARIO DE EQUIPOS DE COMPUTO SEDES EDUCATIVAS.xlsx')
print(f'Cargando: {ruta_equipos}')

df_equipos = pd.read_excel(ruta_equipos)
print(f'Carga exitosa.')
df_equipos.head()

In [ ]:
# Analisis de calidad - Equipos
nulos_equipos = analisis_calidad(df_equipos, 'INVENTARIO DE EQUIPOS DE COMPUTO')

In [ ]:
# Columnas del dataset
print('Columnas disponibles:')
for i, col in enumerate(df_equipos.columns, 1):
    print(f'  {i:2d}. {col} ({df_equipos[col].dtype})')

In [ ]:
# Graficos - Inventario Equipos
grafico_nulos(df_equipos, 'Inventario Equipos')
grafico_tipos_datos(df_equipos, 'Inventario Equipos')
grafico_distribucion_numericas(df_equipos, 'Inventario Equipos')

---
## 6. Intervenciones de Infraestructura

### 6.1 - 22 Sedes con intervencion

In [ ]:
# Cargar 22 sedes
ruta_22 = os.path.join(DATA_PATH, 'Intervenciones de infraestructura', '22 SEDES.xlsx')
print(f'Cargando: {ruta_22}')

df_22_sedes = pd.read_excel(ruta_22)
print(f'Carga exitosa.')
df_22_sedes.head()

In [ ]:
# Analisis de calidad - 22 sedes
nulos_22 = analisis_calidad(df_22_sedes, '22 SEDES CON INTERVENCION')

### 6.2 - 49 Sedes Emprestito (Valores 2026 y 2027)

In [ ]:
# Cargar 49 sedes emprestito
ruta_49 = os.path.join(DATA_PATH, 'Intervenciones de infraestructura', '49 sedes Empr\u00e9stito - Valores 2026 y 2027.xlsx')

if not os.path.exists(ruta_49):
    # Buscar archivo con nombre similar
    carpeta_infra = os.path.join(DATA_PATH, 'Intervenciones de infraestructura')
    archivos = [a for a in os.listdir(carpeta_infra) if '49' in a]
    if archivos:
        ruta_49 = os.path.join(carpeta_infra, archivos[0])

print(f'Cargando: {ruta_49}')
df_49_sedes = pd.read_excel(ruta_49)
print(f'Carga exitosa.')
df_49_sedes.head()

In [ ]:
# Analisis de calidad - 49 sedes
nulos_49 = analisis_calidad(df_49_sedes, '49 SEDES EMPRESTITO - VALORES 2026 Y 2027')

### 6.3 - Historico de Obras 2020-2025

In [ ]:
# Cargar historico obras
ruta_historico = os.path.join(DATA_PATH, 'Intervenciones de infraestructura', 'HISTORICO OBRAS 2020-2025.xlsx')
print(f'Cargando: {ruta_historico}')

df_historico = pd.read_excel(ruta_historico)
print(f'Carga exitosa.')
df_historico.head()

In [ ]:
# Analisis de calidad - Historico obras
nulos_historico = analisis_calidad(df_historico, 'HISTORICO OBRAS 2020-2025')

In [ ]:
# Graficos - Infraestructura consolidado
grafico_nulos(df_22_sedes, '22 Sedes')
grafico_nulos(df_49_sedes, '49 Sedes Emprestito')
grafico_nulos(df_historico, 'Historico Obras')
grafico_distribucion_numericas(df_historico, 'Historico Obras 2020-2025')

---
## 7. Resumen consolidado de calidad de datos

In [ ]:
# Resumen consolidado
resumen = pd.DataFrame({
    'Fuente': [
        'Matricula 2026',
        'Indicadores Eficiencia/Cobertura 2026',
        'Estudiantes por Docente/Equipo 2026',
        'Informacion Geografica Sedes',
        'Inventario Equipos Computo',
        '22 Sedes Intervencion',
        '49 Sedes Emprestito',
        'Historico Obras 2020-2025'
    ],
    'Filas': [
        len(df_matricula),
        len(df_indicadores),
        len(df_docentes),
        len(df_geo),
        len(df_equipos),
        len(df_22_sedes),
        len(df_49_sedes),
        len(df_historico)
    ],
    'Columnas': [
        df_matricula.shape[1],
        df_indicadores.shape[1],
        df_docentes.shape[1],
        df_geo.shape[1],
        df_equipos.shape[1],
        df_22_sedes.shape[1],
        df_49_sedes.shape[1],
        df_historico.shape[1]
    ],
    'Nulos_Total': [
        df_matricula.isnull().sum().sum(),
        df_indicadores.isnull().sum().sum(),
        df_docentes.isnull().sum().sum(),
        df_geo.isnull().sum().sum(),
        df_equipos.isnull().sum().sum(),
        df_22_sedes.isnull().sum().sum(),
        df_49_sedes.isnull().sum().sum(),
        df_historico.isnull().sum().sum()
    ],
    'Duplicados': [
        df_matricula.duplicated().sum(),
        df_indicadores.duplicated().sum(),
        df_docentes.duplicated().sum(),
        df_geo.duplicated().sum(),
        df_equipos.duplicated().sum(),
        df_22_sedes.duplicated().sum(),
        df_49_sedes.duplicated().sum(),
        df_historico.duplicated().sum()
    ]
})

resumen['Pct_Nulos'] = (resumen['Nulos_Total'] / (resumen['Filas'] * resumen['Columnas']) * 100).round(2)

print('=' * 70)
print('RESUMEN CONSOLIDADO DE CALIDAD DE DATOS')
print('=' * 70)
print(resumen.to_string(index=False))

---
## 8. Visualizaciones consolidadas

In [ ]:
# Grafico 1: Registros por fuente (barras)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(resumen['Fuente'], resumen['Filas'], color=sns.color_palette('Set2'))
ax.set_xlabel('Cantidad de registros')
ax.set_title('Registros por fuente de datos')
for bar in bars:
    width = bar.get_width()
    ax.text(width + 5, bar.get_y() + bar.get_height()/2,
            f'{int(width):,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico 2: Porcentaje de nulos por fuente
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#66c2a5' if x < 5 else '#fc8d62' if x < 20 else '#e78ac3' for x in resumen['Pct_Nulos']]
bars = ax.barh(resumen['Fuente'], resumen['Pct_Nulos'], color=colors)
ax.set_xlabel('Porcentaje de nulos (%)')
ax.set_title('Porcentaje de valores nulos por fuente')
ax.axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='Umbral 5%')
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.2, bar.get_y() + bar.get_height()/2,
            f'{width:.1f}%', va='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Grafico 3: Duplicados por fuente
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(resumen['Fuente'], resumen['Duplicados'], color=sns.color_palette('Set2')[3])
ax.set_xlabel('Cantidad de registros duplicados')
ax.set_title('Registros duplicados por fuente')
for bar in bars:
    width = bar.get_width()
    if width > 0:
        ax.text(width + 0.5, bar.get_y() + bar.get_height()/2,
                f'{int(width)}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico 4: Resumen visual de calidad (heatmap simplificado)
calidad_matrix = resumen[['Fuente', 'Pct_Nulos', 'Duplicados']].copy()
calidad_matrix['Pct_Duplicados'] = (resumen['Duplicados'] / resumen['Filas'] * 100).round(2)
calidad_matrix = calidad_matrix.set_index('Fuente')[['Pct_Nulos', 'Pct_Duplicados']]
calidad_matrix.columns = ['% Nulos', '% Duplicados']

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(calidad_matrix, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Porcentaje (%)'})
ax.set_title('Mapa de calidad de datos por fuente')
plt.tight_layout()
plt.show()

---
## 9. Conclusiones del analisis exploratorio

Completar despues de ejecutar todas las celdas con las observaciones principales:

- **Matricula:** [observaciones]
- **Indicadores:** [observaciones]
- **Docentes/Equipos:** [observaciones]
- **Info Geografica:** [observaciones]
- **Inventario Equipos:** [observaciones]
- **Infraestructura:** [observaciones]

### Proximos pasos
1. Estandarizar nombres de sedes/IE entre fuentes
2. Cruzar datos con informacion geografica
3. Calcular indicadores derivados para el ICET
4. Normalizar con ref_min/ref_max fijos
5. Generar visualizaciones territoriales (mapas por comuna)